In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from source.random_walk import generate_bat_flight

#from source.attractor_networks.single_bump import ToroidZhang1996
from source.attractor_networks.grid_bump import ToroidZhang1996

import tqdm
import os

In [ ]:
T = 1000
dt = 0.5e-3
n = 64
net = ToroidZhang1996(n=n, dt=dt,periodicity=18)
net.warm_up()
plt.imshow(net.s)
save_dir = "simulation_data"

In [ ]:
bat_flight = generate_bat_flight(T=T, dt=dt)
recorded_cells = [
    (i, j)
    for i in np.random.randint(0, n - 1, size=3)
    for j in np.random.randint(0, n - 1, size=3)
]

n_steps = bat_flight["time"].shape[0]
n_popul_snapshots = 1000
recording = np.zeros((n_steps, len(recorded_cells)))
population_recordings = np.zeros((n_popul_snapshots, n, n))
popul_snapshots_integers = np.linspace(0, n_steps, n_popul_snapshots, dtype=int)
snapshot_iter = 0
for step_iter in tqdm.tqdm(range(n_steps)):
    recording[step_iter] = np.array(
        [net.s[cell_index] for cell_index in recorded_cells]
    )
    net.step(*bat_flight["dir_vel"][step_iter])
    if step_iter == popul_snapshots_integers[snapshot_iter]:
        population_recordings[snapshot_iter] = net.s
        step_iter += 1

os.makedirs(save_dir, exist_ok=True)
np.save(os.path.join(save_dir, "bat_grid_bump_cell_recording.npy"), recording)
np.save(os.path.join(save_dir, "bat_grid_bump_cell_population.npy"), recording)
np.save(os.path.join(save_dir, "bat_grid_bump_cell_position.npy"), bat_flight["pos"])
np.save(os.path.join(save_dir, "bat_grid_bump_cell_velocity.npy"), bat_flight["vel"])
np.save(os.path.join(save_dir, "bat_grid_bump_cell_direction.npy"), bat_flight["dir"])
np.save(
    os.path.join(save_dir, "bat_grid_bump_cell_turn_velocity.npy"),
    bat_flight["dir_vel"],
)
np.save(os.path.join(save_dir, "bat_grid_bump_cell_time.npy"), bat_flight["time"])

In [ ]:
bat_flight = {}
recording = np.load(os.path.join(save_dir, "bat_grid_bump_cell_recording.npy"))
bat_flight["pos"] = np.load(os.path.join(save_dir, "bat_grid_bump_cell_position.npy"))
bat_flight["vel"] = np.load(os.path.join(save_dir, "bat_grid_bump_cell_velocity.npy"))
bat_flight["dir"] = np.load(os.path.join(save_dir, "bat_grid_bump_cell_direction.npy"))
bat_flight["dir_vel"] = np.load(
    os.path.join(save_dir, "bat_grid_bump_cell_turn_velocity.npy")
)
bat_flight["time"] = np.load(os.path.join(save_dir, "bat_grid_bump_cell_time.npy"))
n_steps = bat_flight["time"].shape[0]

In [ ]:
from source.plot_tools import activity_map
azimuth_activity_map, edges = activity_map(bat_flight["dir"], recording[:, 0], nbins=25)

In [ ]:
plt.imshow(azimuth_activity_map, extent=(edges[0][0],edges[0][-1],edges[1][0], edges[1][-1]),interpolation="bilinear")

In [ ]:
position_activity_map, edges = activity_map(bat_flight["pos"], recording[:, 0], nbins=25)
X,Y,Z = np.meshgrid(*[(edge[:-1] + edge[1:])/2 for edge in edges])
coords_1 = np.stack([X,Y,Z], axis=-1).reshape(-1,3)[:,:,np.newaxis]
coords_2 = np.stack([X,Y,Z], axis=-1).reshape(3,-1)[np.newaxis,:,:]
dist = np.sqrt(np.sum(np.square(coords_1 - coords_2), axis = 1))
weights = position_activity_map.reshape(-1,1) * position_activity_map.reshape(1,-1)
counts,edges = np.histogram(dist.flatten(), weights=weights.flatten())
plt.stairs(counts, edges)